# NB00: Environment Setup and Verification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/glioma-brain-digital-twin-public/blob/main/notebooks/NB00_Setup_and_Verification.ipynb)


## Learning Objectives
- Verify and configure the computational environment for neuroimaging analysis
- Detect GPU availability and optimize compute resources
- Validate all required packages for structural and diffusion MRI analysis
- Explore the project repository structure and BIDS data organization
- Load and inspect a sample NIfTI file metadata

## Overview
This notebook establishes the foundation for the digital twin project by ensuring all dependencies are available and the environment is properly configured for working with brain MRI data.

In [24]:
# Detect environment and set up paths
import os
import sys
from pathlib import Path

# Detect if we're in Google Colab
if 'google.colab' in sys.modules:
    print('Running in Google Colab')
    REPO_ROOT = Path('/content/glioma-brain-digital-twin')
    if not REPO_ROOT.exists():
        import subprocess
        _clone_result = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/arvidl/glioma-brain-digital-twin-public.git',
             '/content/glioma-brain-digital-twin'],
            capture_output=True, text=True)
        if _clone_result.returncode != 0:
            raise RuntimeError(f'git clone failed: {_clone_result.stderr}')
    !pip install -q -r /content/glioma-brain-digital-twin/environment/requirements-colab.txt
    DATA_ROOT = REPO_ROOT / 'data' / 'BIDS_preop_postop'
    if not (DATA_ROOT / 'sub-PAT16').exists():
        !python /content/glioma-brain-digital-twin/data/download_data.py
    os.chdir(REPO_ROOT)
else:
    print('Running locally')
    _cwd = Path.cwd().resolve()
    REPO_ROOT = _cwd if (_cwd / 'pyproject.toml').exists() else _cwd.parent
    DATA_ROOT = REPO_ROOT / 'data' / 'BIDS_preop_postop'

print(f'Repository root: {REPO_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Data exists: {DATA_ROOT.exists()}')
sys.path.insert(0, str(REPO_ROOT))

Running locally
Repository root: <REPO_ROOT>
Data root: <REPO_ROOT>/data/BIDS_preop_postop
Data exists: True


### 1. GPU Detection and CUDA Support

In [25]:
# Check hardware accelerators (GPU / Apple Silicon)
import platform
import numpy as np

print('=' * 60)
print('HARDWARE ACCELERATION')
print('=' * 60)

print(f'Platform:      {platform.platform()}')
print(f'Architecture:  {platform.machine()}')
print(f'NumPy version: {np.__version__}')

# Apple Silicon (MPS)
if platform.machine() == 'arm64' and platform.system() == 'Darwin':
    print(f'\nApple Silicon detected ({platform.processor() or "arm64"})')
    try:
        import torch
        mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
        print(f'  PyTorch version: {torch.__version__}')
        print(f'  MPS (Metal) available: {mps_available}')
        if mps_available:
            print(f'  MPS device: {torch.device("mps")}')
    except ImportError:
        print('  PyTorch not installed (optional -- install with: pip install torch)')

# NVIDIA CUDA
try:
    import torch
    cuda_available = torch.cuda.is_available()
    print(f'\nCUDA available: {cuda_available}')
    if cuda_available:
        print(f'  CUDA version: {torch.version.cuda}')
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1e9:.1f} GB)')
    if not cuda_available and not (platform.machine() == 'arm64' and platform.system() == 'Darwin'):
        print('  No GPU detected. Computations will run on CPU.')
    print(f'  PyTorch version: {torch.__version__}')
except ImportError:
    print('\nPyTorch not installed (optional)')
    print('  Install with: pip install torch torchvision torchaudio')
    print('  All notebooks run without PyTorch.')

HARDWARE ACCELERATION
Platform:      macOS-26.3-arm64-arm-64bit
Architecture:  arm64
NumPy version: 1.26.4

Apple Silicon detected (arm)
  PyTorch version: 2.10.0
  MPS (Metal) available: True
  MPS device: mps

CUDA available: False
  PyTorch version: 2.10.0


### 2. Core Package Verification

In [26]:
# Import and verify core neuroimaging packages
import warnings
warnings.filterwarnings('ignore')

packages_to_check = {
    'nibabel': 'NIfTI file handling',
    'nilearn': 'fMRI/sMRI visualization and analysis',
    'dipy': 'Diffusion imaging and fiber tracking',
    'bids': 'BIDS format utilities',
    'skimage': 'Image processing',
    'scipy': 'Scientific computing',
    'pandas': 'Data manipulation',
    'matplotlib': 'Plotting',
    'seaborn': 'Statistical visualization'
}

print('=' * 60)
print('PACKAGE VERIFICATION')
print('=' * 60)

for package_name, description in packages_to_check.items():
    try:
        pkg = __import__(package_name)
        version = getattr(pkg, '__version__', 'version unknown')
        status = 'OK'
        print(f'{status} {package_name:15} v{version:10} - {description}')
    except ImportError as e:
        print(f'FAIL {package_name:15} - NOT INSTALLED - {description}')

PACKAGE VERIFICATION
OK nibabel         v5.3.2      - NIfTI file handling
OK nilearn         v0.13.1     - fMRI/sMRI visualization and analysis
OK dipy            v1.11.0     - Diffusion imaging and fiber tracking
OK bids            v0.21.0     - BIDS format utilities
OK skimage         v0.26.0     - Image processing
OK scipy           v1.17.1     - Scientific computing
OK pandas          v3.0.1      - Data manipulation
OK matplotlib      v3.10.8     - Plotting
OK seaborn         v0.13.2     - Statistical visualization


### 3. Project Structure Overview

In [27]:
# Display the project directory structure
from pathlib import Path
import os

def show_tree(path, prefix='', max_depth=3, current_depth=0, max_files=20):
    if current_depth >= max_depth:
        return
    
    try:
        items = sorted(os.listdir(path))
    except PermissionError:
        return
    
    dirs = [d for d in items if os.path.isdir(os.path.join(path, d)) and not d.startswith('.')]
    files = [f for f in items if os.path.isfile(os.path.join(path, f)) and not f.startswith('.')]
    
    # Show files
    for i, f in enumerate(files[:max_files]):
        is_last = (i == len(files) - 1) and len(dirs) == 0
        marker = 'L-- ' if is_last else '|-- '
        print(f'{prefix}{marker}{f}')
    
    if len(files) > max_files:
        print(f'{prefix}|-- ... and {len(files) - max_files} more files')
    
    # Show directories
    for i, d in enumerate(dirs):
        is_last = i == len(dirs) - 1
        marker = 'L-- ' if is_last else '|-- '
        print(f'{prefix}{marker}{d}/')
        
        next_prefix = prefix + ('    ' if is_last else '|   ')
        show_tree(os.path.join(path, d), next_prefix, max_depth, current_depth + 1)

print('Project Structure:')
print(f'{REPO_ROOT.name}/')
show_tree(REPO_ROOT, max_depth=2)

Project Structure:
glioma-brain-digital-twin/
|-- BIDS_preop_postop_v1.0.0.zip
|-- BIDS_preop_postop_v2.0.0.zip
|-- CHANGES
|-- Dockerfile
|-- FILES_CREATED.md
|-- INSTALLATION.md
|-- LICENSE
|-- Makefile
|-- README.md
|-- pyproject.toml
|-- MyStuff/
|   L-- BIDS_preop_postop/
|-- beamer/
|   |-- Makefile
|   |-- main-blx.bib
|   |-- main.aux
|   |-- main.bbl
|   |-- main.bcf
|   |-- main.blg
|   |-- main.fdb_latexmk
|   |-- main.fls
|   |-- main.log
|   |-- main.nav
|   |-- main.out
|   |-- main.pdf
|   |-- main.run.xml
|   |-- main.snm
|   |-- main.tex
|   |-- main.toc
|   |-- main.vrb
|   |-- preamble.tex
|   |-- references.bib
|   |-- figures/
|   |-- gslides/
|   L-- sections/
|-- data/
|   |-- download_data.py
|   L-- BIDS_preop_postop/
|-- derivatives/
|   |-- pipeline_qc_checklist.md
|   |-- TVB/
|   |-- connectivity/
|   |-- fmriprep/
|   |-- freesurfer/
|   |-- mrtrix_recon/
|   |-- qsiprep/
|   |-- registered_t1w/
|   |-- resseg/
|   |-- rsHRF/
|   L-- tumorsynth/
|-- docs/


### 4. BIDS Data Overview

In [28]:
# Inspect BIDS data directory structure
if DATA_ROOT.exists():
    print(f'\nBIDS Data Root: {DATA_ROOT}')
    print('\nDirectory Contents:')
    show_tree(DATA_ROOT, max_depth=3)
    
    # List subjects
    subjects = sorted([d for d in os.listdir(DATA_ROOT) if d.startswith('sub-')])
    print(f'\nFound {len(subjects)} subjects: {", ".join(subjects)}')
else:
    print(f'WARNING: Data directory not found at {DATA_ROOT}')
    print('Make sure the data is available in the expected location.')


BIDS Data Root: <REPO_ROOT>/data/BIDS_preop_postop

Directory Contents:
|-- CHANGES
|-- README
|-- dataset_description.json
|-- participants.json
|-- participants.tsv
|-- derivatives/
|   |-- TVB/
|   |   L-- sub-PAT16/
|   |-- rsHRF/
|   |   L-- sub-PAT16/
|   L-- tumor_masks/
|       L-- sub-PAT16/
L-- sub-PAT16/
    |-- sub-PAT16_sessions.json
    |-- sub-PAT16_sessions.tsv
    |-- ses-postop/
    |   |-- anat/
    |   |-- dwi/
    |   L-- func/
    L-- ses-preop/
        |-- anat/
        |-- dwi/
        L-- func/

Found 1 subjects: sub-PAT16


### 5. Load and Inspect Sample NIfTI File

In [29]:
# Load a sample T1w image and inspect its metadata
import nibabel as nib

# Try to load a T1w image from first available subject
sample_nii_path = None

if DATA_ROOT.exists():
    t1_files = sorted(DATA_ROOT.glob('sub-*/ses-*/anat/*T1w.nii.gz'))
    if not t1_files:
        t1_files = sorted(DATA_ROOT.glob('sub-*/anat/*T1w.nii.gz'))
    if t1_files:
        sample_nii_path = t1_files[0]

if sample_nii_path:
    print(f'Loading sample file: {sample_nii_path.name}')
    img = nib.load(sample_nii_path)
    
    print('\n' + '=' * 60)
    print('NIfTI FILE METADATA')
    print('=' * 60)
    
    print(f'\nFile path: {sample_nii_path}')
    print(f'Data shape: {img.shape}')
    print(f'Data dtype: {img.get_data_dtype()}')
    print(f'Affine matrix:\n{img.affine}')
    print(f'\nHeader info:')
    print(f'  Voxel sizes (mm): {img.header.get_zooms()[:3]}')
    print(f'  Data type: {img.header.get_data_dtype()}')
    print(f"  Bit pix: {img.header['bitpix']}")
else:
    print('No sample NIfTI file found to load.')
    print('Ensure BIDS data is present in the expected location.')

Loading sample file: sub-PAT16_ses-postop_T1w.nii.gz

NIfTI FILE METADATA

File path: <REPO_ROOT>/data/BIDS_preop_postop/sub-PAT16/ses-postop/anat/sub-PAT16_ses-postop_T1w.nii.gz
Data shape: (160, 256, 256)
Data dtype: int16
Affine matrix:
[[ 1.00000000e+00  0.00000000e+00 -3.47893895e-03 -7.49530945e+01]
 [ 0.00000000e+00  1.00000000e+00  0.00000000e+00 -1.11718048e+02]
 [ 3.47896013e-03  0.00000000e+00  9.99993920e-01 -1.30906250e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

Header info:
  Voxel sizes (mm): (1.0000061, 1.0, 1.0)
  Data type: int16
  Bit pix: 16


### 6. Verification Summary

In [30]:
# Create a summary of the environment setup
import platform

print('\n' + '=' * 60)
print('ENVIRONMENT VERIFICATION SUMMARY')
print('=' * 60)

# Detect accelerator
_has_accel = False
try:
    import torch
    if torch.cuda.is_available():
        _has_accel = True
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        _has_accel = True
except ImportError:
    pass

checks = {
    'Hardware accelerator (GPU/MPS)': _has_accel,
    'Core packages': True,
    'BIDS data present': DATA_ROOT.exists(),
    'Repository accessible': REPO_ROOT.exists(),
}

for check_name, status in checks.items():
    status_str = 'PASS' if status else 'WARN'
    print(f'{status_str:8} {check_name}')

if not _has_accel:
    print('\n  Note: No GPU/MPS detected. All notebooks run fine on CPU.')
    print('  PyTorch is optional -- install it for GPU-accelerated workflows.')

print('\n' + '=' * 60)
print('Setup complete! Ready to proceed to NB01.')
print('=' * 60)


ENVIRONMENT VERIFICATION SUMMARY
PASS     Hardware accelerator (GPU/MPS)
PASS     Core packages
PASS     BIDS data present
PASS     Repository accessible

Setup complete! Ready to proceed to NB01.


In [31]:
# ── Pipeline Output Verification ─────────────────────────────────────────────
# Check which pipeline steps have been completed (optional -- only relevant
# after running the processing pipeline).

print('=' * 60)
print('Pipeline Output Verification')
print('=' * 60)

_pipeline_checks = {
    'FreeSurfer (pre-op)':   'derivatives/freesurfer/sub-PAT16_ses-preop',
    'FreeSurfer (post-op)':  'derivatives/freesurfer/sub-PAT16_ses-postop',
    'fMRIPrep (pre-op)':     'derivatives/fmriprep/sub-PAT16/ses-preop',
    'fMRIPrep (post-op)':    'derivatives/fmriprep/sub-PAT16/ses-postop',
    'QSIPrep (pre-op)':      'derivatives/qsiprep/sub-PAT16/ses-preop',
    'QSIPrep (post-op)':     'derivatives/qsiprep/sub-PAT16/ses-postop',
    'MRtrix3 (pre-op)':      'derivatives/mrtrix_recon/sub-PAT16/ses-preop',
    'MRtrix3 (post-op)':     'derivatives/mrtrix_recon/sub-PAT16/ses-postop',
    'T1w registration (pre)': 'derivatives/registered_t1w/sub-PAT16/ses-preop',
    'T1w registration (post)':'derivatives/registered_t1w/sub-PAT16/ses-postop',
    'Longitudinal reg':       'derivatives/registered_t1w/sub-PAT16/longitudinal',
    'Pre/post comparison':    'derivatives/registered_t1w/sub-PAT16/preop_vs_postop',
    'SC matrices':            'derivatives/connectivity/sub-PAT16/ses-preop/sc_dk84.npy',
    'FC matrices':            'derivatives/connectivity/sub-PAT16/ses-preop/fc_pearson.npy',
    'rsHRF maps':             'derivatives/rsHRF/sub-PAT16',
    'RESSEG cavity':          'derivatives/resseg/sub-PAT16',
}

_n_done = 0
for _name, _path in _pipeline_checks.items():
    _full = REPO_ROOT / _path
    _ok = _full.exists()
    _n_done += _ok
    _mark = '✓' if _ok else '·'
    print(f'  {_mark}  {_name}')

print(f'\n  Pipeline: {_n_done}/{len(_pipeline_checks)} steps completed')
if _n_done == 0:
    print('  (This is expected if you have not run the processing pipeline.)')
    print('  (Track A notebooks work without pipeline outputs.)')
print('=' * 60)

Pipeline Output Verification
  ✓  FreeSurfer (pre-op)
  ✓  FreeSurfer (post-op)
  ✓  fMRIPrep (pre-op)
  ✓  fMRIPrep (post-op)
  ✓  QSIPrep (pre-op)
  ✓  QSIPrep (post-op)
  ✓  MRtrix3 (pre-op)
  ✓  MRtrix3 (post-op)
  ✓  T1w registration (pre)
  ✓  T1w registration (post)
  ✓  Longitudinal reg
  ✓  Pre/post comparison
  ✓  SC matrices
  ✓  FC matrices
  ✓  rsHRF maps
  ✓  RESSEG cavity

  Pipeline: 16/16 steps completed
